# Exercise 4 : Cleaning and Transforming

Run the first cell : it imports pandas and sets `DATA` to the folder with the exercise files. The
steps use three of them :

- `titanic.csv` : one row per passenger of the Titanic, 1309 rows
- `pittsburgh2013.csv` : daily weather in Pittsburgh in 2013, from Weather Underground
- `pennsylvania2012_turnout.csv` : the election results from Exercise 3

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parents[1] / 'datasets' / 'exercises'

## Step 1 : Dropping

Read `titanic.csv` into `titanic`. Select the `age` and `cabin` columns into a DataFrame `df` and
print its shape. Drop the incomplete rows of `df` with `dropna(how='any')` and print the shape, then
do the same with `how='all'`. Why do the two shapes differ?

Count the missing values in each column of `titanic` with `isna().sum()` and assign the result to
`missing_per_column`. Keep the columns with at most 1000 missing values as `titanic_reduced`, with
`dropna(axis='columns', thresh=...)`. `thresh` is the minimum number of non-missing values, here
`len(titanic) - 1000`. Print the remaining column names. Which columns were dropped?

In [ ]:
titanic = pd.read_csv(DATA / 'titanic.csv')
df = titanic[['age', 'cabin']]
print(df.shape)
print(df.dropna(how='any').shape)
print(df.dropna(how='all').shape)

missing_per_column = titanic.isna().sum()
print(missing_per_column)
titanic_reduced = titanic.dropna(axis='columns', thresh=len(titanic) - 1000)
print(titanic_reduced.columns.tolist())

## Step 2 : Filling

Count the passengers without a port of embarkation in `embarked` with `isna().sum()`. Fill the
gaps with the text `'unknown'` and store the result back in `embarked`. Print
`titanic['embarked'].value_counts()`. Why is `'unknown'` a better fill than the most common port?

In [ ]:
print(titanic['embarked'].isna().sum())
titanic['embarked'] = titanic['embarked'].fillna('unknown')
print(titanic['embarked'].value_counts())

## Step 3 : Apply

Read `pittsburgh2013.csv` into `weather`. Write a function `to_celsius(f)` that converts degrees
Fahrenheit to Celsius with `(f - 32) * 5 / 9`.

Call `.apply(to_celsius)` on the columns `['Mean TemperatureF', 'Mean Dew PointF']` and assign the
result to `df_celsius`. Rename its columns to `['Mean TemperatureC', 'Mean Dew PointC']` and print
`df_celsius.head()`.

`apply` calls `to_celsius` once per column. Call `to_celsius` directly on the same two columns and
compare the values. Why are they the same?

In [ ]:
weather = pd.read_csv(DATA / 'pittsburgh2013.csv')


def to_celsius(f):
    return (f - 32) * 5 / 9


df_celsius = weather[['Mean TemperatureF', 'Mean Dew PointF']].apply(to_celsius)
df_celsius.columns = ['Mean TemperatureC', 'Mean Dew PointC']
print(df_celsius.head())

direct = to_celsius(weather[['Mean TemperatureF', 'Mean Dew PointF']])
print((direct.to_numpy() == df_celsius.to_numpy()).all())

## Step 4 : Map

Read `pennsylvania2012_turnout.csv` into `election` with `index_col='county'`. The dictionary
`red_vs_blue` gives each candidate a color. Map the `winner` column with it into a new column
`color` and print `election.head()`.

Map `sample_winners` with the same dictionary and print the result. Which values become `NaN`? Why
is that safer than a rule that returns `'red'` for every name other than `'Obama'`?

In [ ]:
red_vs_blue = {'Obama': 'blue', 'Romney': 'red'}
sample_winners = pd.Series(['Obama', 'Romney', 'Unknown', None])

In [ ]:
election = pd.read_csv(DATA / 'pennsylvania2012_turnout.csv', index_col='county')
election['color'] = election['winner'].map(red_vs_blue)
print(election.head())

sample = sample_winners.map(red_vs_blue)
print(sample)

## Step 5 : Z-scores

A z-score is the number of standard deviations that a value lies above the mean. Add a column
`turnout_zscore` with the z-score of `turnout`, computed with arithmetic on whole columns and
without `apply`. Print the five counties with the highest z-score. Does a high z-score prove that a
turnout figure is wrong?

In [ ]:
turnout = election['turnout']
election['turnout_zscore'] = (turnout - turnout.mean()) / turnout.std()
print(election.nlargest(5, 'turnout_zscore')[['turnout', 'turnout_zscore']])

## Step 6 : Text

One column name of `weather` starts with a space : print `weather.columns.tolist()` to find it.
Strip the spaces from all column names with `weather.columns.str.strip()`.

`Events` describes the weather of each day, such as `'Rain'` or `'Fog-Snow'`, and is missing for a
quiet day. Count the days with snow with `str.contains('Snow')` and `sum()`. Fill the missing events
with `'None'` and print the `value_counts()` of the column.

In [ ]:
print(weather.columns.tolist())
weather.columns = weather.columns.str.strip()

print(weather['Events'].str.contains('Snow').sum())
weather['Events'] = weather['Events'].fillna('None')
print(weather['Events'].value_counts())

## Step 7 : Types

Create a table `daily` with the `Date`, `Max TemperatureF`, `Min TemperatureF` and `Events` columns
of `weather`. Rename the temperature columns to `max_f` and `min_f`.

Convert `Date` from text to dates with `pd.to_datetime()` and check the result with `daily.dtypes`.
Add a column `range_f`, the difference between `max_f` and `min_f`, and print the three days with
the largest range with `nlargest`.

In [ ]:
daily = weather[['Date', 'Max TemperatureF', 'Min TemperatureF', 'Events']]
daily = daily.rename(columns={'Max TemperatureF': 'max_f', 'Min TemperatureF': 'min_f'})
daily['Date'] = pd.to_datetime(daily['Date'])
print(daily.dtypes)

daily['range_f'] = daily['max_f'] - daily['min_f']
print(daily.nlargest(3, 'range_f'))

## If time permits :

- On how many days with snow did the maximum temperature rise above freezing, 32 °F? What share of
  all days with snow is that?

In [ ]:
snow = weather['Events'].str.contains('Snow')
above_freezing = weather.loc[snow, 'Max TemperatureF'] > 32
print(snow.sum(), above_freezing.sum())
print(round(above_freezing.mean(), 2))